# 5. Model compression

EEEM075 - AI and Sustainability coursework

Applies **two compression techniques to each** tuned model from Notebooks 3-4, and re-evaluates every variant on the same held-out test set (2015) with the same metrics used in Notebook 4 - RMSE/MAE/R², disk size, and inference latency. This is the sustainability payoff of the whole project: a smaller, cheaper-to-run model is directly a more sustainable one, provided it doesn't lose much accuracy to get there.

- **MLP:** structured pruning (physically remove the least useful neurons, not just zero them out), then post-training dynamic quantisation (float32 -> int8 weights).
- **XGBoost:** ensemble truncation (keep only the first K of the tuned model's trees), then knowledge distillation (train a much smaller student model to mimic the tuned model's predictions).

In [14]:
import json
import time
import os
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

import xgboost as xgb

ARTIFACT_DIR = Path('../artifacts')
MODEL_DIR = ARTIFACT_DIR / 'models'

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

FEATURES = ['AT', 'AP', 'AH', 'AFDP', 'GTEP', 'TIT', 'TAT', 'TEY', 'CDP']
TARGET = 'NOX'

In [15]:
train = pd.read_csv(ARTIFACT_DIR / 'train.csv')
val = pd.read_csv(ARTIFACT_DIR / 'val.csv')
test = pd.read_csv(ARTIFACT_DIR / 'test.csv')

X_train, y_train = train[FEATURES].values, train[TARGET].values
X_val, y_val = val[FEATURES].values, val[TARGET].values
X_test, y_test = test[FEATURES].values, test[TARGET].values

scaler = joblib.load(MODEL_DIR / 'scaler.joblib')
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

def evaluate(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'{label:>22} | RMSE: {rmse:.3f}  MAE: {mae:.3f}  R2: {r2:.3f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

baseline_row = pd.read_csv(ARTIFACT_DIR / 'test_efficiency_results.csv', index_col=0)
print('Baseline (from Notebook 4):')
print(baseline_row)

Baseline (from Notebook 4):
              rmse        mae        r2  n_params_or_trees      disk_kb  \
MLP      11.688527  10.181241 -0.102544            11649.0    49.102539   
XGBoost  14.270012  13.125931 -0.643331              300.0  1780.340820   

         latency_ms  
MLP        0.001723  
XGBoost    0.010863  


## Load the tuned models (uncompressed baselines)

In [16]:
class MLP(nn.Module):
    def __init__(self, n_features, hidden=(64, 32), dropout=0.1):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


with open(MODEL_DIR / 'mlp_tuned_params.json') as f:
    mlp_params = json.load(f)
mlp_params['hidden'] = tuple(mlp_params['hidden'])

mlp_baseline = MLP(n_features=len(FEATURES), hidden=mlp_params['hidden'], dropout=mlp_params['dropout'])
mlp_baseline.load_state_dict(torch.load(MODEL_DIR / 'mlp_tuned.pt'))
mlp_baseline.eval()

with open(MODEL_DIR / 'xgb_tuned_params.json') as f:
    xgb_params = json.load(f)

xgb_baseline = xgb.XGBRegressor()
xgb_baseline.load_model(str(MODEL_DIR / 'xgb_tuned.json'))

print('MLP hidden layers:', mlp_params['hidden'])
print('XGBoost trees:', xgb_baseline.get_booster().num_boosted_rounds())

MLP hidden layers: (128, 64, 32)
XGBoost trees: 300


In [17]:
def model_size_latency(predict_fn, X, disk_bytes, n_repeats=5):
    """Shared measurement helper so every variant (compressed or not) is timed the same way."""
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        _ = predict_fn(X)
        times.append((time.perf_counter() - t0) / len(X))
    return {'disk_kb': disk_bytes / 1024, 'latency_ms': np.mean(times) * 1000}

## MLP - Technique 1: structured pruning

**Why structured, not just magnitude/unstructured pruning:** zeroing out individual weights (unstructured pruning) doesn't actually shrink a dense PyTorch tensor on disk or speed up a plain matrix multiply - the zeros are still stored and multiplied. To get a genuinely smaller, faster model (not just a cosmetically sparse one), this removes entire neurons - ranked by the L1 norm of their outgoing weights, lowest importance first - and physically rebuilds smaller `Linear` layers from the survivors.

A short fine-tuning pass follows, since removing neurons outright typically costs some accuracy - a brief retrain at a low learning rate recovers most of it, which is standard practice for pruning.

In [18]:
PRUNE_RATIO = 0.5  # fraction of neurons removed from each hidden layer

def structured_prune_mlp(model, hidden, dropout, prune_ratio=0.5):
    linear_layers = [m for m in model.net if isinstance(m, nn.Linear)]
    hidden_layers, output_layer = linear_layers[:-1], linear_layers[-1]

    new_linears = []
    new_hidden = []
    prev_keep_idx = None  # None = keep all of the original input features

    for layer in hidden_layers:
        W = layer.weight.data.clone()
        b = layer.bias.data.clone()
        if prev_keep_idx is not None:
            W = W[:, prev_keep_idx]
        importance = W.abs().sum(dim=1)  # L1 norm per output neuron
        n_keep = max(1, int(round(W.shape[0] * (1 - prune_ratio))))
        keep_idx = torch.argsort(importance, descending=True)[:n_keep].sort().values

        new_layer = nn.Linear(W.shape[1], n_keep)
        new_layer.weight.data = W[keep_idx, :].clone()
        new_layer.bias.data = b[keep_idx].clone()
        new_linears.append(new_layer)
        new_hidden.append(n_keep)
        prev_keep_idx = keep_idx

    out_W = output_layer.weight.data[:, prev_keep_idx].clone()
    out_b = output_layer.bias.data.clone()
    new_output_layer = nn.Linear(out_W.shape[1], out_W.shape[0])
    new_output_layer.weight.data = out_W
    new_output_layer.bias.data = out_b

    pruned = MLP(n_features=new_linears[0].in_features, hidden=tuple(new_hidden), dropout=dropout)
    pruned_linears = [m for m in pruned.net if isinstance(m, nn.Linear)]
    for target, source in zip(pruned_linears[:-1], new_linears):
        target.weight.data = source.weight.data.clone()
        target.bias.data = source.bias.data.clone()
    pruned_linears[-1].weight.data = new_output_layer.weight.data.clone()
    pruned_linears[-1].bias.data = new_output_layer.bias.data.clone()

    return pruned, tuple(new_hidden)


mlp_pruned, pruned_hidden = structured_prune_mlp(mlp_baseline, mlp_params['hidden'], mlp_params['dropout'], PRUNE_RATIO)
print('Original hidden sizes:', mlp_params['hidden'], '-> Pruned hidden sizes:', pruned_hidden)

mlp_pruned_n_params_before_finetune = sum(p.numel() for p in mlp_pruned.parameters())
print('Parameters after pruning (before fine-tune):', mlp_pruned_n_params_before_finetune)

Original hidden sizes: (128, 64, 32) -> Pruned hidden sizes: (64, 32, 16)
Parameters after pruning (before fine-tune): 3265


In [19]:
def to_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def finetune(model, epochs=20, lr=1e-4, patience=5, batch_size=256):
    """Continues training an already-initialised model (e.g. post-pruning) at a low learning rate,
    to recover accuracy lost from structural changes. Same early-stopping-on-val-loss pattern as
    the original training loop in Notebook 3."""
    train_loader = to_loader(X_train_scaled, y_train, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break

    model.load_state_dict(best_state)
    return model


mlp_pruned = finetune(mlp_pruned)
print('Fine-tuning complete.')

Fine-tuning complete.


In [20]:
torch.save(mlp_pruned.state_dict(), MODEL_DIR / 'mlp_pruned.pt')
mlp_pruned_disk_bytes = os.path.getsize(MODEL_DIR / 'mlp_pruned.pt')
mlp_pruned_n_params = sum(p.numel() for p in mlp_pruned.parameters())

mlp_pruned.eval()
with torch.no_grad():
    mlp_pruned_pred = mlp_pruned(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()
mlp_pruned_metrics = evaluate(y_test, mlp_pruned_pred, 'MLP (pruned, test)')

def mlp_pruned_predict(x_scaled):
    with torch.no_grad():
        return mlp_pruned(torch.tensor(x_scaled, dtype=torch.float32)).numpy()

mlp_pruned_perf = model_size_latency(mlp_pruned_predict, X_test_scaled, mlp_pruned_disk_bytes)
print(f'Params: {mlp_pruned_n_params:,} (baseline had {sum(p.numel() for p in mlp_baseline.parameters()):,})')
print(mlp_pruned_perf)

    MLP (pruned, test) | RMSE: 12.114  MAE: 10.123  R2: -0.184
Params: 3,265 (baseline had 11,649)
{'disk_kb': 16.3662109375, 'latency_ms': np.float64(0.0004952302272456778)}


## MLP - Technique 2: post-training dynamic quantisation

Converts the (unpruned) baseline MLP's `Linear` layer weights from 32-bit floats to 8-bit integers after training - no retraining needed, applied directly to the already-tuned baseline. This is a different, complementary compression strategy to pruning: pruning reduces *how many* numbers the model stores, quantisation reduces *how many bits* each number takes. PyTorch's dynamic quantisation is CPU-only, which matches how this model will run in a real deployment anyway (no GPU dependency for such a small network).

In [21]:
mlp_baseline_cpu = copy.deepcopy(mlp_baseline).cpu().eval()
mlp_quantized = torch.quantization.quantize_dynamic(
    mlp_baseline_cpu, {nn.Linear}, dtype=torch.qint8,
)

torch.save(mlp_quantized.state_dict(), MODEL_DIR / 'mlp_quantized.pt')
mlp_quantized_disk_bytes = os.path.getsize(MODEL_DIR / 'mlp_quantized.pt')

with torch.no_grad():
    mlp_quantized_pred = mlp_quantized(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()
mlp_quantized_metrics = evaluate(y_test, mlp_quantized_pred, 'MLP (quantized, test)')

def mlp_quantized_predict(x_scaled):
    with torch.no_grad():
        return mlp_quantized(torch.tensor(x_scaled, dtype=torch.float32)).numpy()

mlp_quantized_perf = model_size_latency(mlp_quantized_predict, X_test_scaled, mlp_quantized_disk_bytes)
print(mlp_quantized_perf)

C:\Users\Acer\AppData\Local\Temp\ipykernel_22676\3807562099.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  mlp_quantized = torch.quantization.quantize_dynamic(


 MLP (quantized, test) | RMSE: 11.639  MAE: 10.125  R2: -0.093
{'disk_kb': 18.2412109375, 'latency_ms': np.float64(0.001424493499323201)}


## MLP compression summary

In [22]:
# Baseline row pulled directly from Notebook 4's saved results (same model, no need to recompute)
mlp_baseline_row = {k: baseline_row.loc['MLP', k] for k in ['rmse', 'mae', 'r2', 'disk_kb', 'latency_ms']}

mlp_comparison = pd.DataFrame({
    'MLP (baseline)': mlp_baseline_row,
    'MLP (pruned)': {**mlp_pruned_metrics, **mlp_pruned_perf},
    'MLP (quantized)': {**mlp_quantized_metrics, **mlp_quantized_perf},
}).T

print(mlp_comparison)
mlp_comparison.to_csv(ARTIFACT_DIR / 'mlp_compression_results.csv')

                      rmse        mae        r2    disk_kb  latency_ms
MLP (baseline)   11.688527  10.181241 -0.102544  49.102539    0.001723
MLP (pruned)     12.113815  10.123267 -0.184236  16.366211    0.000495
MLP (quantized)  11.638912  10.125258 -0.093204  18.241211    0.001424


## XGBoost - Technique 1: ensemble truncation

Keeps only the first *K* trees of the already-trained, tuned ensemble (a direct slice of the trained booster, not a retrain)- a form of pruning appropriate to boosted trees, where 'weight magnitude' pruning doesn't apply the way it does to a neural net. Truncating a boosted ensemble is a meaningful test of how much the later trees are actually contributing, since gradient boosting adds trees that correct the residual error of all previous trees - later trees usually contribute less than earlier ones.

In [23]:
TRUNCATE_FRACTION = 0.5  # keep this fraction of the trained trees

xgb_booster = xgb_baseline.get_booster()
n_trees_total = xgb_booster.num_boosted_rounds()
n_keep = max(1, int(round(n_trees_total * TRUNCATE_FRACTION)))

xgb_truncated_booster = xgb_booster[:n_keep]
xgb_truncated_booster.save_model(str(MODEL_DIR / 'xgb_truncated.json'))
xgb_truncated_disk_bytes = os.path.getsize(MODEL_DIR / 'xgb_truncated.json')

dtest = xgb.DMatrix(X_test)
xgb_truncated_pred = xgb_truncated_booster.predict(dtest)
xgb_truncated_metrics = evaluate(y_test, xgb_truncated_pred, 'XGBoost (truncated, test)')

def xgb_truncated_predict(x):
    return xgb_truncated_booster.predict(xgb.DMatrix(x))

xgb_truncated_perf = model_size_latency(xgb_truncated_predict, X_test, xgb_truncated_disk_bytes)
print(f'Trees: {n_trees_total} -> {n_keep}')
print(xgb_truncated_perf)

XGBoost (truncated, test) | RMSE: 13.782  MAE: 12.644  R2: -0.533
Trees: 300 -> 150
{'disk_kb': 940.486328125, 'latency_ms': np.float64(0.004009739978250039)}


## XGBoost - Technique 2: knowledge distillation

Trains a much smaller 'student' XGBoost (fewer, shallower trees) to mimic the tuned model's *predictions* rather than the original ground truth - the standard distillation setup. This is a distinct compression strategy from truncation: instead of cutting down the existing ensemble, it learns a fresh, deliberately small model whose target is the teacher's output.

One caveat: since the teacher itself doesn't generalise well to 2015 (the concept-drift finding from Notebook 4), the student is distilled from an imperfect teacher and inherits similar limitations on the test set.

In [24]:
teacher_train_pred = xgb_baseline.predict(X_train)

xgb_student_params = dict(n_estimators=30, max_depth=3, learning_rate=0.1)
xgb_student = xgb.XGBRegressor(random_state=SEED, n_jobs=-1, **xgb_student_params)
xgb_student.fit(X_train, teacher_train_pred)  # distillation: fit on the teacher's predictions

xgb_student.save_model(str(MODEL_DIR / 'xgb_student.json'))
xgb_student_disk_bytes = os.path.getsize(MODEL_DIR / 'xgb_student.json')

xgb_student_pred = xgb_student.predict(X_test)
xgb_student_metrics = evaluate(y_test, xgb_student_pred, 'XGBoost (student, test)')

def xgb_student_predict(x):
    return xgb_student.predict(x)

xgb_student_perf = model_size_latency(xgb_student_predict, X_test, xgb_student_disk_bytes)
print(f'Student trees: {xgb_student.get_booster().num_boosted_rounds()} (vs. {n_trees_total} in teacher)')
print(xgb_student_perf)

XGBoost (student, test) | RMSE: 13.139  MAE: 11.904  R2: -0.393
Student trees: 30 (vs. 300 in teacher)
{'disk_kb': 34.4560546875, 'latency_ms': np.float64(0.0007624322849826308)}


## XGBoost compression summary

In [25]:
# Baseline row pulled directly from Notebook 4's saved results (same model, no need to recompute)
xgb_baseline_row = {k: baseline_row.loc['XGBoost', k] for k in ['rmse', 'mae', 'r2', 'disk_kb', 'latency_ms']}

xgb_comparison = pd.DataFrame({
    'XGBoost (baseline)': xgb_baseline_row,
    'XGBoost (truncated)': {**xgb_truncated_metrics, **xgb_truncated_perf},
    'XGBoost (student)': {**xgb_student_metrics, **xgb_student_perf},
}).T

print(xgb_comparison)
xgb_comparison.to_csv(ARTIFACT_DIR / 'xgb_compression_results.csv')

                          rmse        mae        r2      disk_kb  latency_ms
XGBoost (baseline)   14.270012  13.125931 -0.643331  1780.340820    0.010863
XGBoost (truncated)  13.781708  12.644084 -0.532789   940.486328    0.004010
XGBoost (student)    13.139404  11.904254 -0.393245    34.456055    0.000762


## Combined summary - everything, side by side

Every variant of both models, on the same test set and with the same metrics. None of the R² values are strong in absolute terms because of the concept-drift limitation discussed in Notebook 4, so the informative comparison is the relative one: each compressed variant against its own uncompressed baseline.

In [26]:
all_results = pd.concat([mlp_comparison, xgb_comparison])
print(all_results)
all_results.to_csv(ARTIFACT_DIR / 'all_compression_results.csv')

                          rmse        mae        r2      disk_kb  latency_ms
MLP (baseline)       11.688527  10.181241 -0.102544    49.102539    0.001723
MLP (pruned)         12.113815  10.123267 -0.184236    16.366211    0.000495
MLP (quantized)      11.638912  10.125258 -0.093204    18.241211    0.001424
XGBoost (baseline)   14.270012  13.125931 -0.643331  1780.340820    0.010863
XGBoost (truncated)  13.781708  12.644084 -0.532789   940.486328    0.004010
XGBoost (student)    13.139404  11.904254 -0.393245    34.456055    0.000762


**Findings:**

- Structured pruning removed ~72% of the MLP's parameters (11,649 → 3,265), cut its disk size from 49 KB to 16 KB and roughly halved its per-sample latency, at a small accuracy cost (test RMSE 11.69 → 12.11).
- Dynamic quantisation was near-lossless (test RMSE 11.69 → 11.64, marginally better) while shrinking the model to 18 KB with a slightly lower latency - the cleanest accuracy/size trade-off of the four techniques.
- Truncating the XGBoost ensemble to 150 trees roughly halved its disk size, cut its latency by ~2.6×, and slightly *improved* test RMSE (14.27 → 13.78): the later trees were fitting 2011-2013 patterns that do not transfer to 2015.
- The 30-tree distilled student is ~52× smaller than the teacher (34 KB vs 1.8 MB), ~3× faster per prediction, and also the most accurate XGBoost variant on test (RMSE 13.14) - again consistent with the full ensemble overfitting the training years.
- Overall, every compressed variant kept or improved test accuracy while cutting size and latency. For XGBoost, compression effectively doubled as regularisation under drift - the smaller, cheaper models are not just adequate here, they are better.